<a href="https://colab.research.google.com/github/Leashaniya/Research-Project/blob/leasha/01_pastpaper_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Past Paper Pipeline
Hybrid OCR + diagrams → blueprint + subquestions + chunks

In [ ]:
# ==========================================================
# NOTEBOOK 1 — PAST PAPER PIPELINE (BOXED-ONLY INPUT)
#
# ✅ Update in this version (FIXES your Q5 marks issue):
#   - Total marks for a main question is now extracted robustly even when
#     the header like "(20 Marks)" appears ABOVE the "Question 5" line
#     due to a page break.
#   - Prevents mistakenly taking subquestion marks (e.g., 2, 3, 8) as the
#     main question total.
#
# Still true:
#   - Processes ONLY PDFs in: past_paper_red_box
#   - Uses the SAME boxed PDF for:
#       (1) text extraction (PDF text layer / OCR fallback)
#       (2) red-box detection to crop diagrams
#   - Any text that falls INSIDE a detected red box region is excluded.
#   - Subquestion extraction remains DISABLED (PP1 requirement).
#
# Outputs per PDF:
#   text_extraction_hybrid/<pdf_stem>/pages_text/*.txt
#   text_extraction_hybrid/<pdf_stem>/diagrams/* (cropped red-box diagrams)
#   text_extraction_hybrid/<pdf_stem>/all_text_with_diagrams.txt
#   text_extraction_hybrid/<pdf_stem>/cleaned_document.txt
#   text_extraction_hybrid/<pdf_stem>/blueprint.json
#   text_extraction_hybrid/<pdf_stem>/blueprint_with_subquestions.json (empty subq arrays)
#
# Global outputs:
#   text_extraction_hybrid/chunks.jsonl
#   text_extraction_hybrid/chunks_index.csv
#   text_extraction_hybrid/diagrams_manifest.json
#
# QC outputs:
#   _tmp_pdf_pages_hybrid/ocr_qc_report/ocr_conf_report.json
#   _tmp_pdf_pages_hybrid/ocr_qc_report/flagged_pages.txt
# ==========================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# -------------------------------
# 1) System deps
# -------------------------------
!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr tesseract-ocr-eng
!pip install --quiet pymupdf pdf2image pytesseract tqdm opencv-python-headless

# -------------------------------
# 2) Imports
# -------------------------------
from pathlib import Path
import shutil, os, re, json, csv, time
from tqdm import tqdm
import fitz
import cv2
import numpy as np
import pytesseract
from pdf2image import convert_from_path
from pytesseract import Output

# -------------------------------
# 3) CONFIG
# -------------------------------
ROOT_PDFS = Path("/content/drive/MyDrive/RP/past_paper_red_box")
OUT_ROOT  = Path("/content/drive/MyDrive/RP/text_extraction_hybrid")
TMP_PAGES = Path("/content/drive/MyDrive/RP/_tmp_pdf_pages_hybrid")

DPI = 300
FALLBACK_DPI = 600
TESS_CONFIG = "--oem 3 --psm 6"
SCALE_FACTOR = 1.6
DO_DESKEW = True

RED_RANGES = [
    (np.array([0, 80, 50]), np.array([10, 255, 255])),
    (np.array([170, 80, 50]), np.array([180, 255, 255]))
]
MIN_AREA = 2000
PAD = 8
THUMB_W = 512

CONF_THRESHOLD = 60
MIN_WORDS_FOR_TEXT = 12
MAX_SINGLE_BOX_FRAC = 0.70

CHUNK_WORDS = 400
OVERLAP_WORDS = 80

MIN_QUESTION_WORDS = 5
SKIP_FIRST_PAGE = True

SKIP_FILES = {
    # "some_bad_file.pdf",
}

OUT_ROOT.mkdir(parents=True, exist_ok=True)
TMP_PAGES.mkdir(parents=True, exist_ok=True)

# -------------------------------
# 4) Utilities
# -------------------------------

def pdf_to_images(pdf_path: Path, out_dir: Path, dpi=DPI):
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in out_dir.glob("*.png"):
        try:
            f.unlink()
        except:
            pass
    pages = convert_from_path(str(pdf_path), dpi=dpi, fmt="png")
    out = []
    for i, p in enumerate(pages, start=1):
        ppath = out_dir / f"page_{i:03d}.png"
        p.save(str(ppath), "PNG")
        out.append(ppath)
    return out


def merge_rects(rects, gap_thresh=12):
    if not rects:
        return []
    rects = sorted(rects, key=lambda r: (r[1], r[0]))
    merged = [list(rects[0])]
    for x, y, w, h in rects[1:]:
        x1, y1, w1, h1 = merged[-1]
        if x <= x1 + w1 + gap_thresh and y <= y1 + h1 + gap_thresh:
            nx = min(x, x1)
            ny = min(y, y1)
            nx2 = max(x + w, x1 + w1)
            ny2 = max(y + h, y1 + h1)
            merged[-1] = [nx, ny, nx2 - nx, ny2 - ny]
        else:
            merged.append([x, y, w, h])
    return [(int(x), int(y), int(w), int(h)) for x, y, w, h in merged]


def detect_red_boxes(img_bgr, ranges=RED_RANGES, pad=PAD, min_area=MIN_AREA):
    if img_bgr is None:
        return []
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask_total = np.zeros(hsv.shape[:2], dtype='uint8')
    for low, high in ranges:
        mask_total = cv2.bitwise_or(mask_total, cv2.inRange(hsv, low, high))

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    mask_total = cv2.morphologyEx(mask_total, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask_total = cv2.dilate(mask_total, np.ones((3, 3), np.uint8), iterations=1)

    contours, _ = cv2.findContours(mask_total, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rects = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w * h < min_area:
            continue
        rects.append((x, y, w, h))

    merged = merge_rects(rects)

    out = []
    for i, (x, y, w, h) in enumerate(sorted(merged, key=lambda r: (r[1], r[0])), start=1):
        x0, y0 = max(0, x - pad), max(0, y - pad)
        x1, y1 = min(img_bgr.shape[1], x + w + pad), min(img_bgr.shape[0], y + h + pad)
        out.append({
            'x': int(x0), 'y': int(y0),
            'w': int(x1 - x0), 'h': int(y1 - y0),
            'idx': i,
            'y_mid': int(y0 + (y1 - y0) // 2)
        })
    return out


def detect_and_deskew(img):
    try:
        osd = pytesseract.image_to_osd(img)
        rot = 0
        for line in osd.splitlines():
            if "Rotate:" in line:
                rot = int(line.split(":")[1].strip())
                break
        if rot != 0:
            h, w = img.shape[:2]
            M = cv2.getRotationMatrix2D((w / 2, h / 2), -rot, 1.0)
            img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    except Exception:
        pass
    return img


def preprocess_variants(img, scale=SCALE_FACTOR):
    out = {}

    img_s = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(img_s, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    g = clahe.apply(gray)
    th = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)
    out['clahe_adapt'] = th

    _, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    out['otsu'] = otsu

    out['scaled_gray'] = gray
    out['otsu_inv'] = 255 - otsu

    img_big = cv2.resize(img, None, fx=2.2, fy=2.2, interpolation=cv2.INTER_CUBIC)
    grayb = cv2.cvtColor(img_big, cv2.COLOR_BGR2GRAY)
    _, otsu_b = cv2.threshold(grayb, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    out['big_otsu'] = otsu_b

    return out


def ocr_best_variant(img, config=TESS_CONFIG):
    variants = preprocess_variants(img)
    best_txt, best_score, best_name = "", -1e9, None

    for name, vimg in variants.items():
        data = pytesseract.image_to_data(vimg, config=config, output_type=Output.DICT)
        confs = []
        for c in data.get('conf', []):
            try:
                ci = float(c)
                if ci >= 0:
                    confs.append(ci)
            except:
                pass
        mean_conf = float(np.mean(confs)) if confs else -1

        txt = pytesseract.image_to_string(vimg, config=config)
        score = mean_conf + 0.001 * len(txt)

        if score > best_score:
            best_score, best_txt, best_name = score, txt, name

    return {"text": best_txt, "score": best_score, "variant": best_name}


def is_inside_any_y(yv, diagrams):
    for d in diagrams:
        if d['y'] <= yv <= (d['y'] + d['h']):
            return True
    return False

# -------------------------------
# 5) Process one PDF (boxed-only)
# -------------------------------

def process_one_pdf(pdf_path: Path, dpi_used=DPI):
    stem = pdf_path.stem
    print("\nProcessing:", pdf_path.name, "(dpi", dpi_used, ")")

    out_pdf_dir = OUT_ROOT / stem
    pages_out = out_pdf_dir / "pages_text"
    diagrams_out = out_pdf_dir / "diagrams"
    pages_out.mkdir(parents=True, exist_ok=True)
    diagrams_out.mkdir(parents=True, exist_ok=True)

    try:
        doc = fitz.open(str(pdf_path))
    except Exception as e:
        print(" ⚠️ can't open PDF:", e)
        return False

    n_pages = doc.page_count

    boxed_dir = TMP_PAGES / (stem + "_boxed")
    if boxed_dir.exists():
        shutil.rmtree(boxed_dir)
    pdf_to_images(pdf_path, boxed_dir, dpi=dpi_used)

    combined_text = ""

    page_indices = list(range(1, n_pages + 1))
    if SKIP_FIRST_PAGE and page_indices:
        page_indices = page_indices[1:]

    for page_num in page_indices:
        print(" Page:", page_num)
        page = doc[page_num - 1]

        img_bgr = None
        candidate = boxed_dir / f"page_{page_num:03d}.png"
        if candidate.exists():
            img_bgr = cv2.imread(str(candidate))

        if img_bgr is None:
            pix = page.get_pixmap(dpi=dpi_used)
            tmp_path = TMP_PAGES / f"{stem}_page_{page_num:03d}.png"
            pix.save(str(tmp_path))
            img_bgr = cv2.imread(str(tmp_path))

        diagrams = detect_red_boxes(img_bgr)

        # Try PDF text layer first
        lines = []
        use_ocr = True
        try:
            blocks = page.get_text("blocks")
            text_blocks = [b for b in blocks if len(b) > 4 and str(b[4]).strip()]

            if text_blocks:
                w_img, h_img = img_bgr.shape[1], img_bgr.shape[0]
                w_pdf, h_pdf = page.rect.width, page.rect.height
                sx, sy = w_img / w_pdf, h_img / h_pdf

                for b in text_blocks:
                    x0, y0, x1, y1, txt = b[:5]
                    y_img = int(y0 * sy)
                    if not is_inside_any_y(y_img, diagrams):
                        lines.append({'y': y_img, 'text': str(txt).strip()})

                lines = sorted(lines, key=lambda r: r['y'])
                page_text = "\n".join([ln['text'] for ln in lines]).strip()
                use_ocr = False
            else:
                page_text = ""
        except Exception:
            page_text = ""
            use_ocr = True

        # OCR fallback
        mean_conf = 95.0
        if use_ocr:
            mask = np.ones(img_bgr.shape[:2], dtype=np.uint8) * 255
            for d in diagrams:
                cv2.rectangle(mask, (d['x'], d['y']), (d['x'] + d['w'], d['y'] + d['h']), 0, -1)
            img_masked = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)

            if DO_DESKEW:
                img_masked = detect_and_deskew(img_masked)

            _ = ocr_best_variant(img_masked, config=TESS_CONFIG)
            data = pytesseract.image_to_data(img_masked, config=TESS_CONFIG, output_type=Output.DICT)

            ocr_lines = []
            confs = []
            n = len(data.get('text', []))
            for i in range(n):
                txt = str(data['text'][i]).strip()
                if not txt:
                    continue

                try:
                    y = int(data['top'][i])
                except:
                    y = None

                if y is None or not is_inside_any_y(y, diagrams):
                    ocr_lines.append({'y': y if y is not None else 0, 'text': txt})

                try:
                    ci = float(data['conf'][i])
                    if ci >= 0:
                        confs.append(ci)
                except:
                    pass

            mean_conf = float(np.mean(confs)) if confs else -1

            lines.extend(ocr_lines)
            lines = sorted(lines, key=lambda r: (r['y'] or 0))
            page_text = "\n".join([ln['text'] for ln in lines]).strip()

        # Save diagrams and inject placeholders
        for d in diagrams:
            name = f"page_{page_num:03d}_diagram_{d['idx']}.png"
            crop = img_bgr[d['y']:d['y'] + d['h'], d['x']:d['x'] + d['w']]
            cv2.imwrite(str(diagrams_out / name), crop)

            h_crop, w_crop = crop.shape[:2]
            new_w = THUMB_W
            new_h = max(1, int(h_crop * (THUMB_W / max(1, w_crop))))
            thumb = cv2.resize(crop, (new_w, new_h), interpolation=cv2.INTER_AREA)
            cv2.imwrite(str(diagrams_out / name.replace(".png", "_thumb.png")), thumb)

            lines.append({'y': int(d['y_mid']), 'text': f"[DIAGRAM: {name}]"})

        lines = sorted(lines, key=lambda r: (r['y'] or 0))
        page_text = "\n".join([ln['text'] for ln in lines]).strip()

        page_file = pages_out / f"page_{page_num:03d}_text.txt"
        page_file.write_text(page_text or "", encoding="utf-8")

        combined_text += f"\n\n--- PAGE {page_num} ---\n{page_text}"
        print(f" -> words: {len((page_text or '').split())} | diagrams: {len(diagrams)} | mean_conf:{mean_conf:.1f}")

    (out_pdf_dir / "all_text_with_diagrams.txt").write_text(combined_text, encoding="utf-8")
    print(" Saved:", out_pdf_dir / "all_text_with_diagrams.txt")
    return True

# -------------------------------
# 6) OCR QC
# -------------------------------

def run_ocr_qc(tmp_pages_root=TMP_PAGES, out_dir=None,
               conf_threshold=CONF_THRESHOLD,
               min_words_for_text=MIN_WORDS_FOR_TEXT,
               max_single_box_frac=MAX_SINGLE_BOX_FRAC):
    if out_dir is None:
        out_dir = tmp_pages_root / "ocr_qc_report"
    out_dir.mkdir(parents=True, exist_ok=True)

    report = []
    bad = []

    for img_path in sorted(tmp_pages_root.rglob("page_*.png")):
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        h, w = img.shape[:2]
        page_area = float(w * h) if (w * h) > 0 else 1.0

        diagrams = detect_red_boxes(img)

        max_single_box_area = 0.0
        for d in diagrams:
            max_single_box_area = max(max_single_box_area, (d['w'] * d['h']))
        page_max_single_box_frac = (max_single_box_area / page_area) if page_area > 0 else 0.0

        data = pytesseract.image_to_data(img, config=TESS_CONFIG, output_type=Output.DICT)
        confs_filtered = []
        n_words = 0

        n_items = len(data.get('text', []))
        for i in range(n_items):
            txt = str(data['text'][i]).strip()
            if not txt:
                continue

            try:
                y_top = int(data['top'][i])
            except:
                y_top = None

            if y_top is not None and is_inside_any_y(y_top, diagrams):
                continue

            n_words += 1

            try:
                ci = float(data['conf'][i])
                if ci >= 0:
                    confs_filtered.append(ci)
            except:
                pass

        mean_conf = float(np.mean(confs_filtered)) if confs_filtered else -1

        rec = {
            "page": str(img_path),
            "mean_conf": mean_conf,
            "n_words": n_words,
            "max_single_box_frac": page_max_single_box_frac
        }
        report.append(rec)

        if page_max_single_box_frac >= max_single_box_frac:
            rec["flagged_reason"] = "diagram_only"
            continue
        if n_words < min_words_for_text:
            rec["flagged_reason"] = "diagram_only"
            continue

        if mean_conf < conf_threshold:
            rec["flagged_reason"] = "low_conf"
            bad.append(rec)

    with open(out_dir / "ocr_conf_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    with open(out_dir / "flagged_pages.txt", "w", encoding="utf-8") as f:
        for r in bad:
            f.write(
                f"{r['page']} | mean_conf={r['mean_conf']} | n_words={r.get('n_words',0)} | "
                f"max_single_box_frac={r.get('max_single_box_frac',0):.3f}\n"
            )

    print("OCR QC Done. Flagged pages (excluding diagram-only):", len(bad))
    return report, bad

# -------------------------------
# 7) Diagrams manifest
# -------------------------------

def build_diagrams_manifest(out_root=OUT_ROOT):
    manifest_path = out_root / "diagrams_manifest.json"
    man = []

    for pdf_dir in out_root.iterdir():
        if not pdf_dir.is_dir():
            continue

        diag_folder = pdf_dir / "diagrams"
        if not diag_folder.exists():
            continue

        for f in diag_folder.glob("page_*_diagram_*.png"):
            if f.name.endswith("_thumb.png"):
                continue

            m = re.search(r'page[_\-]?0*([0-9]+)', f.name, re.IGNORECASE)
            page_no = int(m.group(1)) if m else None

            man.append({
                "pdf_stem": pdf_dir.name,
                "page_no": page_no,
                "diagram_file": str(f),
                "thumb_file": str(f).replace(".png", "_thumb.png")
            })

    with open(manifest_path, "w", encoding="utf-8") as mf:
        json.dump(man, mf, indent=2)

    print("Diagrams manifest written:", manifest_path)
    return manifest_path

# -------------------------------
# 8) Blueprint parser (MAIN questions only) — UPDATED MARKS LOGIC
# -------------------------------

Q_MAIN_RE = re.compile(r'.*\bQuestion\s*([0-9IVXLC]+)\b', re.IGNORECASE)
NUMERIC_MAIN_RE = re.compile(r'^\s*\(?\s*([0-9]+)\s*(?:[\.\)\-:])\s*', re.IGNORECASE)
ALT_Q_RE = re.compile(r'.*\bQ\s*[:\.]?\s*([0-9]+)\b', re.IGNORECASE)

# NOTE: keep MARKS_RE broad for subquestions, but we will NOT use first match as total.
MARKS_RE = re.compile(r'\(?\s*([0-9]{1,3})\s*marks?\s*\)?', re.IGNORECASE)

FOOTER_RE = re.compile(r'Page\s+\d+\s*of\s*\d+', re.I)
HEADER_MARK_RE = re.compile(r'(Question\s+\d+)\s*\(\s*\d+\s*marks?\s*\)', re.I)

# NEW: strict total marks pattern that matches only "(20 Marks)" style
TOTAL_MARKS_PAREN_RE = re.compile(r"\(\s*(\d{1,3})\s*Marks?\s*\)", re.IGNORECASE)


def clean_question_text(text: str) -> str:
    text = FOOTER_RE.sub('', text)
    text = HEADER_MARK_RE.sub(r'\1', text)
    return text.strip()


def _looks_like_question_total(n: int) -> bool:
    # Typical totals (your papers are usually 20). Allow 10-100.
    return 10 <= n <= 100


def extract_total_marks_for_question(q_lines, prev_page_tail_lines):
    """
    Robust extraction:
      1) Look for '(xx Marks)' in the first few lines of the question (header zone)
      2) If not found, look in previous page tail lines (page-break header)
      3) If still not found, fallback to summing marks found in the question block
         (use ONLY if sum is a sane total)
    """
    header_zone = "\n".join(q_lines[:6])  # first ~6 lines is usually enough
    m = TOTAL_MARKS_PAREN_RE.search(header_zone)
    if m:
        val = int(m.group(1))
        if _looks_like_question_total(val):
            return val

    prev_zone = "\n".join(prev_page_tail_lines[-10:]) if prev_page_tail_lines else ""
    m2 = TOTAL_MARKS_PAREN_RE.search(prev_zone)
    if m2:
        val = int(m2.group(1))
        if _looks_like_question_total(val):
            return val

    # fallback: sum of subquestion marks
    all_marks = [int(x) for x in MARKS_RE.findall("\n".join(q_lines))]
    s = sum(all_marks)
    if _looks_like_question_total(s):
        return s

    return None


def parse_blueprint_from_text(doc_text: str, pdf_stem: str, min_words=MIN_QUESTION_WORDS):
    blueprint = []
    state = {"current_q": None, "q_buffer": [], "q_page": None}

    # Track tail of previous page (for page-break marks like your Q5)
    prev_page_tail = []
    current_page_lines = []

    def finalize():
        if state["current_q"]:
            q_text = "\n".join(state["q_buffer"]).strip()
            if not q_text:
                return

            q_lines = state["q_buffer"]
            total_marks = extract_total_marks_for_question(q_lines, prev_page_tail)

            diag_refs = [d.strip() for d in re.findall(r'\[DIAGRAM:([^\]]+)\]', q_text)]

            blueprint.append({
                "question_id": str(state["current_q"]),
                "pdf_stem": pdf_stem,
                "page_no": state["q_page"],
                "marks": int(total_marks) if total_marks is not None else None,
                "text": q_text,
                "diagram_refs": diag_refs
            })

        state["current_q"] = None
        state["q_buffer"] = []

    page_no = None

    for ln in doc_text.splitlines():
        ln_strip = ln.strip()

        # page marker
        mpage = re.search(r'---\s*PAGE\s*([0-9]+)\s*---', ln_strip, re.IGNORECASE)
        if mpage:
            # when entering a new page, set prev_page_tail from previous page's lines
            prev_page_tail = current_page_lines[-20:] if current_page_lines else prev_page_tail
            current_page_lines = []

            page_no = int(mpage.group(1))
            continue

        # accumulate for tail tracking
        current_page_lines.append(ln_strip)

        if not ln_strip:
            if state["current_q"]:
                state["q_buffer"].append("")
            continue

        m_main = Q_MAIN_RE.match(ln_strip) or NUMERIC_MAIN_RE.match(ln_strip) or ALT_Q_RE.match(ln_strip)
        if m_main:
            finalize()
            qid = m_main.group(1)
            state["current_q"] = qid
            state["q_page"] = page_no
            state["q_buffer"] = [ln_strip]
            continue

        if state["current_q"]:
            state["q_buffer"].append(ln_strip)

    finalize()
    blueprint = [b for b in blueprint if len((b.get("text") or "").split()) >= (min_words or 1)]
    return blueprint

# -------------------------------
# 9) Build cleaned_document + blueprints + chunks
# -------------------------------

def build_cleaned_docs_blueprints_and_chunks(out_root=OUT_ROOT):
    PAGE_TEXT_GLOB = "*/pages_text/page_*_text.txt"

    chunks_jsonl = out_root / "chunks.jsonl"
    chunks_csv = out_root / "chunks_index.csv"

    files = sorted(out_root.glob(PAGE_TEXT_GLOB))
    pages_by_pdf = {}

    for f in files:
        stem = f.parent.parent.name
        m = re.search(r'page[_\-]?0*([0-9]+)', f.name, re.IGNORECASE)
        page_no = int(m.group(1)) if m else None
        txt = f.read_text(encoding="utf-8", errors="ignore")
        pages_by_pdf.setdefault(stem, []).append((page_no or 0, txt, str(f)))

    global_chunks = []
    csv_rows = []

    for pdf_stem, page_list in pages_by_pdf.items():
        page_list = sorted(page_list, key=lambda x: x[0])

        doc_lines = []
        for pn, txt, _ in page_list:
            doc_lines.append(f"\n\n--- PAGE {pn} ---\n")
            s = txt.replace("\x0c", " ")
            s = re.sub(r'[^\x00-\x7F]+', ' ', s)
            doc_lines.append(s)

        doc_text = "".join(doc_lines).strip()

        out_pdf_dir = out_root / pdf_stem
        cleaned_path = out_pdf_dir / "cleaned_document.txt"
        cleaned_path.write_text(doc_text, encoding="utf-8")

        blueprint = parse_blueprint_from_text(doc_text, pdf_stem)

        # Subquestions disabled
        for b in blueprint:
            b['subquestions_full'] = []
            b['subquestions'] = []

        blueprint_main = []
        for b in blueprint:
            b2 = dict(b)
            b2.pop('subquestions', None)
            b2.pop('subquestions_full', None)
            blueprint_main.append(b2)

        with open(out_pdf_dir / "blueprint.json", "w", encoding="utf-8") as bf:
            json.dump(blueprint_main, bf, indent=2, ensure_ascii=False)

        with open(out_pdf_dir / "blueprint_with_subquestions.json", "w", encoding="utf-8") as bf:
            json.dump(blueprint, bf, indent=2, ensure_ascii=False)

        # chunking
        words = re.sub(r'\n', ' \n ', doc_text).split()
        n = len(words)
        step = CHUNK_WORDS - OVERLAP_WORDS

        start = 0
        c_i = 0
        while start < n:
            end = min(start + CHUNK_WORDS, n)
            chunk_words = words[start:end]
            chunk_text = " ".join(chunk_words).replace(" \n ", "\n").strip()

            chunk_id = f"{pdf_stem}__c{c_i:04d}"
            page_matches = re.findall(r'--- PAGE (\d+) ---', chunk_text)
            chunk_page = int(page_matches[0]) if page_matches else None

            rec = {
                "chunk_id": chunk_id,
                "pdf_stem": pdf_stem,
                "page_no": chunk_page,
                "start_word": start,
                "end_word": end,
                "n_words": len(chunk_words),
                "text": chunk_text,
                "source": str(cleaned_path)
            }
            global_chunks.append(rec)

            csv_rows.append({
                "chunk_id": chunk_id,
                "pdf_stem": pdf_stem,
                "page_no": chunk_page,
                "start_word": start,
                "end_word": end,
                "n_words": len(chunk_words),
                "text_snippet": (chunk_text[:200] + "...") if len(chunk_text) > 200 else chunk_text,
            })

            c_i += 1
            start += step

    with open(chunks_jsonl, "w", encoding="utf-8") as jf:
        for rec in global_chunks:
            jf.write(json.dumps(rec, ensure_ascii=False) + "\n")

    with open(chunks_csv, "w", newline="", encoding="utf-8") as cf:
        writer = csv.DictWriter(cf, fieldnames=list(csv_rows[0].keys()) if csv_rows else ["chunk_id"])
        writer.writeheader()
        writer.writerows(csv_rows)

    print("DONE: chunks written:", len(global_chunks))
    return True

# -------------------------------
# 10) Driver
# -------------------------------

def main_full_run(max_passes: int = 2):
    pdfs = sorted(ROOT_PDFS.glob("*.pdf"))
    pdfs = [p for p in pdfs if p.name not in SKIP_FILES]

    print("Found boxed PDFs:", len(pdfs))

    for p in pdfs:
        process_one_pdf(p, dpi_used=DPI)

    run_ocr_qc()
    build_diagrams_manifest()
    build_cleaned_docs_blueprints_and_chunks()

    for i in range(max_passes):
        print(f"\n=== AUTO QC PASS {i+1}/{max_passes} ===")
        _, bad = run_ocr_qc()
        if not bad:
            print("No low-confidence pages. Done.")
            break

        print("Low-confidence pages:", len(bad), "-> re-render all boxed PDFs at", FALLBACK_DPI)

        for p in pdfs:
            process_one_pdf(p, dpi_used=FALLBACK_DPI)

        time.sleep(1)
        build_diagrams_manifest()
        build_cleaned_docs_blueprints_and_chunks()

    print("\nPipeline finished. Outputs in:", OUT_ROOT)

# -------------------------------
# 11) RUN
# -------------------------------
main_full_run(max_passes=2)


# ==========================================================
# QC SCRIPT (unchanged) — optional run after pipeline
# ==========================================================
from pathlib import Path
import json, re

ROOT_PDFS = Path("/content/drive/MyDrive/RP/past_paper_red_box")
OUT_ROOT  = Path("/content/drive/MyDrive/RP/text_extraction_hybrid")

DIAG_TAG_RE = re.compile(r"\[DIAGRAM:\s*([^\]]+)\]")

def read_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def qc_one(pdf_path: Path):
    stem = pdf_path.stem
    out_dir = OUT_ROOT / stem

    pages_dir = out_dir / "pages_text"
    diags_dir = out_dir / "diagrams"

    all_text = out_dir / "all_text_with_diagrams.txt"
    cleaned  = out_dir / "cleaned_document.txt"
    bp_main  = out_dir / "blueprint.json"

    issues = []

    required = [pages_dir, diags_dir, all_text, cleaned, bp_main]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        issues.append(("missing_outputs", missing))
        return issues

    page_files = sorted(pages_dir.glob("page_*_text.txt"))
    diag_files = sorted([p for p in diags_dir.glob("page_*_diagram_*.png") if not p.name.endswith("_thumb.png")])

    txt = all_text.read_text(encoding="utf-8", errors="ignore")
    placeholders = [m.strip() for m in DIAG_TAG_RE.findall(txt)]

    missing_from_disk = [fn for fn in placeholders if not (diags_dir / fn).exists()]
    if missing_from_disk:
        issues.append(("placeholders_point_to_missing_files", missing_from_disk[:20]))

    bp = read_json(bp_main)
    if not isinstance(bp, list) or len(bp) == 0:
        issues.append(("blueprint_empty_or_invalid", bp_main.as_posix()))
    else:
        bad_refs = []
        marks_sum = 0
        missing_marks = 0

        for q in bp:
            m = q.get("marks", None)
            if m is None:
                missing_marks += 1
            else:
                try:
                    marks_sum += int(m)
                except:
                    missing_marks += 1

            for dr in (q.get("diagram_refs") or []):
                if not (diags_dir / dr).exists():
                    bad_refs.append(dr)

        if bad_refs:
            issues.append(("blueprint_bad_diagram_refs", bad_refs[:20]))

        issues.append(("summary", {
            "pages_text_files": len(page_files),
            "diagrams_saved": len(diag_files),
            "placeholders_found": len(placeholders),
            "blueprint_questions": len(bp),
            "marks_sum_known_only": marks_sum,
            "missing_marks_count": missing_marks
        }))

    return issues


def run_qc():
    pdfs = sorted(ROOT_PDFS.glob("*.pdf"))
    print("QC running on PDFs:", len(pdfs))
    print("-" * 80)

    ok = 0
    warn = 0

    for p in pdfs:
        issues = qc_one(p)

        summary = None
        for t, v in issues:
            if t == "summary":
                summary = v

        hard_errors = [t for t, _ in issues if t in (
            "missing_outputs",
            "blueprint_empty_or_invalid",
            "placeholders_point_to_missing_files",
            "blueprint_bad_diagram_refs"
        )]
        status = "OK" if not hard_errors else "CHECK"

        if status == "OK":
            ok += 1
        else:
            warn += 1

        print(f"{status}: {p.name}")
        if summary:
            print("  ", summary)

        for t, v in issues:
            if t != "summary" and t != "none":
                print(f"  - {t}: {v}")
        print("-" * 80)

    print("DONE")
    print("  OK:", ok)
    print("  CHECK:", warn)

# Uncomment if you want QC immediately after pipeline:
# run_qc()


Mounted at /content/drive
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 117528 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 36.3 MB/s eta 0:00:00
Found boxed PDFs: 15

Processing: 2014 (1).pdf (dpi 300 )
 Page: 2
 -> words: 357 | diagrams: 0 | mean_conf:95.0
 Page: 3
 -> words: 344 | diagrams: 1 | mean_conf:95.0
 Page: 4
 -> words: 147 | diagrams: 2 | mean_conf:95.0
 Page: 5
 -> words: 229 | diagrams: 0 | mean_conf:95.0
 Page: 6
 -> words: 272 | diagrams: 1 | mean_conf:95.0
 Saved: /content/drive/MyD